## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from feature_engine.outliers import Winsorizer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier, StackingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, classification_report,
                             confusion_matrix, RocCurveDisplay)

## Importing DataSet

In [ ]:
df = pd.read_csv('../Facebook_Ads_2.csv', encoding='ISO-8859-1')
df.head()
df.tail()
df['Country'].unique()

## EDA (Exploratory Data Analysis)

In [ ]:
click = df[df['Clicked'] == 1]
no_click = df[df['Clicked'] == 0]

print('Total =', len(df))
print('Number of Customers who Clicked on the Ad -', len(click))
print('CTR -', 1. * len(click) / len(df) * 100.0, '%')              # Click Through Rate
print('No Click -', len(no_click))
print('Non Click Rate -', 1. * len(no_click) / len(df) * 100.0, '%')

plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='Clicked')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')
plt.title("Count of Click")
plt.show()

### Distributions and associations of attributes (распределения и связи признаков)

### Boxplot

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.boxplot(x='Clicked', y='Salary', data=df)
plt.title('Salary & Clicked')                     
plt.subplot(1, 2, 2)
sns.boxplot(x='Clicked', y='Time Spent on Site', data=df)
plt.title('Time Spent & Click')
plt.show()

### Scatterplot

In [ ]:
plt.figure(figsize=(13, 5))
sns.scatterplot(x=df['Time Spent on Site'], y=df['Salary'], hue=df['Clicked'])
plt.title('Time Spent on Site & Salary')
plt.show()

### Histplot

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['Salary'], bins=40, kde=True)
plt.title('Salary Distribution')
plt.subplot(1, 2, 2)
sns.histplot(df['Time Spent on Site'], bins=20, kde=True)
plt.title('Time Spent on Site Distribution')
plt.tight_layout()
plt.show()

## Preprocessing

### Splitting the Data

In [ ]:
df.drop(['Names', 'emails', 'Country'], axis=1, inplace=True)

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(['Clicked'], axis=1),
    df.Clicked,
    test_size=0.3,
    random_state=0
)

X_train.shape, X_test.shape

## Pipeline 

### Winsorizer & Scaler

In [ ]:
def make_preprocess_pipe(model):
    return Pipeline([
        ('winsorizer', Winsorizer(capping_method='gaussian', tail='both', fold=3)),
        ('scaler', StandardScaler()),
        ('model', model)
    ])

## Baseline Models

In [ ]:
baseline_models = [
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('svc', SVC(probability=True, random_state=0)),
    ('ada', AdaBoostClassifier(n_estimators=100, random_state=0)),
]


def evaluate_models(models, X_train, X_test, y_train, y_test):
    results = []
    for name, model in models:
        pipe = make_preprocess_pipe(model)
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
        results.append({
            'Model': name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'F1-Score': f1_score(y_test, y_pred, average='binary'),
            'Precision': precision_score(y_test, y_pred, average='binary'),
            'Recall': recall_score(y_test, y_pred, average='binary'),
            'CV_Accuracy': cv_scores.mean(),
        })
    return pd.DataFrame(results)


results_df = evaluate_models(baseline_models, X_train, X_test, y_train, y_test)
print(results_df.set_index('Model').round(4))

## Hyperparametr Tuning

In [ ]:
knn_pipe = make_preprocess_pipe(KNeighborsClassifier())
knn_param_dist = {
    'model__n_neighbors': randint(1, 30),
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2],
}
knn_search = RandomizedSearchCV(knn_pipe, knn_param_dist,
                                n_iter=20, cv=5, scoring='accuracy',
                                random_state=42, n_jobs=-1)
knn_search.fit(X_train, y_train)
print('KNeighbors. best parameters:', knn_search.best_params_)

ada_pipe = make_preprocess_pipe(AdaBoostClassifier(random_state=0))
ada_param_dist = {
    'model__n_estimators': randint(50, 200),
    'model__learning_rate': uniform(0.01, 0.99),
}
ada_search = RandomizedSearchCV(ada_pipe, ada_param_dist,
                                n_iter=30, cv=5, scoring='accuracy',
                                random_state=42, n_jobs=-1)
ada_search.fit(X_train, y_train)
print('AdaBoost. best parameters:', ada_search.best_params_)

rf_pipe = make_preprocess_pipe(RandomForestClassifier(random_state=0))
rf_param_dist = {
    'model__n_estimators': randint(50, 200),
    'model__max_depth': randint(3, 15),
    'model__criterion': ['gini', 'entropy'],
}
rf_search = RandomizedSearchCV(rf_pipe, rf_param_dist,
                               n_iter=20, cv=5, scoring='accuracy',
                               random_state=42, n_jobs=-1)
rf_search.fit(X_train, y_train)
print('RandomForest. best parameters:', rf_search.best_params_)

## StackingClassifier

In [ ]:
strip = lambda params: {k.replace('model__', ''): v for k, v in params.items()}

estimators = [
    ('knn', make_preprocess_pipe(KNeighborsClassifier(**strip(knn_search.best_params_)))),
    ('ada', make_preprocess_pipe(AdaBoostClassifier(random_state=0, **strip(ada_search.best_params_)))),
    ('rf',  make_preprocess_pipe(RandomForestClassifier(random_state=0, **strip(rf_search.best_params_)))),
]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=make_preprocess_pipe(RandomForestClassifier(random_state=0)),
    cv=3, n_jobs=-1
)
stacking_clf.fit(X_train, y_train)

## Model Evaluation

In [ ]:
def evaluate(name, y_test, y_pred, y_proba):
    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred, average='binary'), 4),
        'Precision': round(precision_score(y_test, y_pred, average='binary'), 4),
        'Recall': round(recall_score(y_test, y_pred, average='binary'), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4),
    }


y_pred_knn = knn_search.predict(X_test)
y_pred_ada = ada_search.predict(X_test)
y_pred_rf = rf_search.predict(X_test)
y_pred_stack = stacking_clf.predict(X_test)

y_proba_knn = knn_search.predict_proba(X_test)[:, 1]
y_proba_ada = ada_search.predict_proba(X_test)[:, 1]
y_proba_rf = rf_search.predict_proba(X_test)[:, 1]
y_proba_stack = stacking_clf.predict_proba(X_test)[:, 1]

results = pd.DataFrame([
    evaluate('KNN', y_test, y_pred_knn, y_proba_knn),
    evaluate('AdaBoost', y_test, y_pred_ada, y_proba_ada),
    evaluate('RandomForest', y_test, y_pred_rf, y_proba_rf),
    evaluate('Stacking', y_test, y_pred_stack, y_proba_stack),
]).set_index('Model')

print(results)

best = results['ROC-AUC'].idxmax()
print(f"\nbest parameters ROC-AUC: {best} — {results.loc[best, 'ROC-AUC']}")

## Confusion Matrix

In [ ]:
def plot_confusion_matrix(ax, y_test, y_pred, title):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                cbar=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

fig, axes = plt.subplots(2, 2, figsize=(11, 10))
axes = axes.flatten() 

plot_confusion_matrix(axes[0], y_test, y_pred_knn,   "KNN")
plot_confusion_matrix(axes[1], y_test, y_pred_ada,   "AdaBoost")
plot_confusion_matrix(axes[2], y_test, y_pred_rf,    "Random Forest")
plot_confusion_matrix(axes[3], y_test, y_pred_stack, "StackingClassifier")

plt.tight_layout()
plt.show()

## ROC Curves Analysis

In [ ]:
plt.figure(figsize=(8, 6))
for name, y_proba in [('KNN', y_proba_knn), ('AdaBoost', y_proba_ada),
                      ('RandomForest', y_proba_rf), ('Stacking', y_proba_stack)]:
    RocCurveDisplay.from_predictions(y_test, y_proba, name=name, ax=plt.gca())
plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)')
plt.title('ROC Curves — Facebook Ads Click Prediction')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
## Results

In [ ]:
comparison = results.reset_index().melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='Score', hue='Metric', data=comparison, palette='Set1')
for p in plt.gca().patches:
    height = p.get_height()
    if not np.isnan(height):
        plt.gca().annotate(f'{height:.3f}',
                           (p.get_x() + p.get_width() / 2., height),
                           ha='center', va='bottom', fontsize=8, fontweight='bold')
plt.ylim(0, 1.05)
plt.title('Models Metric', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Лучшие модели по каждой метрике
for metric in ['Accuracy', 'F1-Score', 'Precision', 'Recall', 'ROC-AUC']:
    row = results[metric].idxmax()
    print(f"Best Model {metric}: {row} — {results.loc[row, metric]}")